In [3]:
from sls_client import get_sls_data_by_query
from datetime import datetime

sql="""
type:a |SELECT 
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    date_format(__time__, '%Y%m%d') as ds,regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1) as variant,uid,
    count(1) as requests_cnt,count(distinct uid) users,date_format(min(__time__),'%H:%i') start_time
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{[^}]+\}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"frequent_sku_pool202506"%'
GROUP BY 1,2,3,4 order by 3 LIMIT 1000000
"""

from_time=datetime.strptime(datetime.now().strftime("%Y-%m-%d 00:00:00"),"%Y-%m-%d %H:%M:%S")
to_time=datetime.now()

user_df = get_sls_data_by_query(
    from_time=from_time,
    to_time=to_time,
    query=sql,
    project="xianmu-front-end-log",
    logstore="xm-mall",
)

user_df.describe()

即将获取数据: =====> 2025-08-15 00:00:00 2025-08-15 18:39:50.797404 xm-mall: 
type:a |SELECT 
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id
>=====数条数:11764


,experiment_id,ds,variant,uid,requests_cnt,users,start_time,__source__,__time__
count,11764,11764,11764,11764,11764,11764,11764,11764,11764
unique,1,1,4,11763,918,1,976,1,1
top,frequent_sku_pool202506,20250815,V3,,21,1,13:06,,1755187200
freq,11764,11764,4683,2,134,11764,34,11764,11764


In [ ]:
view_sql="""
type:view and pageName:"/purchase-assistant" and linkInfo:purchaseAssistant| 
select uid,uName,type,pageName,date_format(__time__, '%Y%m%d') as ds,count(*)view_times 
from log group by 1,2,3,4,5 limit 1000000
"""
